# 백투영 (Back Projection)

## 개요

백투영(Back Projection)은 **특정 색상이 이미지의 어느 위치에 분포하는지**를 시각화하는 기법이다. OpenCV의 MeanShift 추적 알고리즘에서 **추적 대상의 위치를 찾기 위한 핵심 전처리 단계**로 사용된다.

## 동작 원리

백투영은 다음 순서로 동작한다.

1. **관심 영역(ROI) 선택** — 추적하고 싶은 객체 영역을 지정한다.
2. **히스토그램 계산** — ROI의 색상 분포(주로 HSV의 H 채널)를 히스토그램으로 만든다.
3. **백투영 수행** — 전체 이미지의 각 픽셀을 히스토그램과 대조하여, 해당 픽셀의 색상 값이 히스토그램에서 얼마나 높은 빈도를 가지는지를 결과 이미지에 기록한다.

결과적으로 **ROI와 비슷한 색상을 가진 픽셀은 밝게, 그렇지 않은 픽셀은 어둡게** 나타나는 확률 맵(probability map)이 생성된다.

## MeanShift와의 관계

MeanShift 알고리즘은 **밀도가 높은 방향으로 탐색 윈도우를 이동**시키는 방식으로 동작한다. 이때 "밀도"의 기준이 되는 것이 바로 백투영 결과 이미지이다.

```
[원본 프레임] → [HSV 변환] → [백투영] → [확률 맵] → [MeanShift 적용] → [객체 위치 추적]
```

- 백투영 결과에서 **밝은 값이 밀집된 영역** = 추적 대상이 있을 확률이 높은 영역
- MeanShift는 이 확률 맵 위에서 윈도우를 반복 이동시켜 **확률이 가장 높은 중심점**을 찾는다.

## OpenCV 코드 예시

```python
import cv2
import numpy as np

# 1. 첫 프레임에서 추적 대상 ROI 설정
ret, frame = cap.read()
roi = frame[y:y+h, x:x+w]

# 2. ROI의 HSV 히스토그램 계산
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
roi_hist = cv2.calcHist([hsv_roi], [0], None, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# 3. 매 프레임마다 백투영 수행
hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
back_proj = cv2.calcBackProject([hsv_frame], [0], roi_hist, [0, 180], 1)

# 4. MeanShift로 추적
track_window = (x, y, w, h)
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
ret, track_window = cv2.meanShift(back_proj, track_window, term_crit)
```

## 핵심 함수: `cv2.calcBackProject()`

| 매개변수 | 설명 |
|----------|------|
| `images` | 입력 이미지 리스트 (HSV 변환된 프레임) |
| `channels` | 사용할 채널 인덱스 (H 채널 → `[0]`) |
| `hist` | ROI에서 계산한 히스토그램 |
| `ranges` | 히스토그램의 값 범위 (`[0, 180]` for H 채널) |
| `scale` | 출력 스케일 팩터 (보통 `1`) |

## 정리

- 백투영은 "이 색상이 이미지 어디에 있는가?"에 대한 답을 **확률 맵**으로 제공한다.
- MeanShift는 이 확률 맵을 입력으로 받아 **객체의 현재 위치를 추적**한다.
- 따라서 백투영의 품질이 추적 정확도에 직접적으로 영향을 미친다.